In [1]:
from dotenv import load_dotenv
load_dotenv()
 
import os
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
 
# ----------------------------
# LLM Setup
# ----------------------------
llm = ChatAnthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    model=os.getenv("LLM_MODEL"),
    base_url=os.getenv("LLM_ENDPOINT")
)

/home/ubuntu/Desktop/GenAi/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ----------------------------
# Simulated Data
# ----------------------------
WEATHER_DATA = {
    "paris": "Sunny, 22°C",
    "tokyo": "Cloudy, 18°C",
    "new york": "Rainy, 15°C"
}
 
ATTRACTIONS = {
    "paris": ["Eiffel Tower", "Louvre Museum", "Notre-Dame"],
    "tokyo": ["Tokyo Tower", "Senso-ji Temple", "Shibuya Crossing"],
    "new york": ["Statue of Liberty", "Central Park", "Times Square"]
}
 
FLIGHTS = {
    ("kolkata", "paris"): "Flight AI101 - 10:30 AM",
    ("kolkata", "tokyo"): "Flight AI202 - 8:00 PM",
    ("kolkata", "new york"): "Flight AI303 - 6:45 AM"
}
 

In [3]:
 
# ----------------------------
# Tools
# ----------------------------
 
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    return WEATHER_DATA.get(city.lower(), "Weather data not available.")
 
 
@tool
def search_flights(from_city: str, to_city: str, date: str) -> str:
    """Search available flights."""
    flight = FLIGHTS.get((from_city.lower(), to_city.lower()))
    if flight:
        return f"{flight} on {date}"
    return "No flights found."
 
 
@tool
def get_attractions(city: str) -> str:
    """Get popular tourist attractions in a city."""
    places = ATTRACTIONS.get(city.lower())
    if places:
        return ", ".join(places)
    return "No attractions found."
 
 

In [4]:
# ----------------------------
# Tools List
# ----------------------------
tools = [get_weather, search_flights, get_attractions]

In [5]:
# ----------------------------
# System Prompt
# ----------------------------
system_prompt = """
You are a Travel Planning Assistant.
 
You can:
- Check weather.
- Search flights.
- Suggest tourist attractions.
 
Always provide clear and friendly travel advice.
"""

In [6]:
# ----------------------------
# Create Agent
# ----------------------------
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

In [7]:
# ----------------------------
# Chat Function
# ----------------------------
def chat(user_input):
    result = agent.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    return result["messages"][-1].content
 

In [8]:
# ----------------------------
# Test
# ----------------------------
print("=" * 60)
print("Travel Planning Agent")
print("=" * 60)
 
response = chat(
    "I'm planning a trip to Paris next week. What's the weather like and what should I see?"
)
 
print(response)

Travel Planning Agent
Great news about your Paris trip! Here's what I found:

**Weather:** 
The weather looks beautiful in Paris next week with **sunny skies and 22°C (about 72°F)**. This is perfect weather for sightseeing! I'd recommend comfortable walking clothes and perhaps a light jacket for the evenings.

**Must-See Attractions:**
Paris has some incredible sights to visit:
- **Eiffel Tower** - The iconic symbol of Paris. Don't miss the views from the top!
- **Louvre Museum** - Home to the Mona Lisa and thousands of other masterpieces. Plan to spend several hours here.
- **Notre-Dame** - The magnificent Gothic cathedral (currently under restoration, but worth viewing the exterior)

**Additional Tips:**
- The sunny weather is perfect for strolling along the Seine River
- Visit cafés to experience the Parisian lifestyle
- Consider booking attractions in advance to avoid long queues

Would you like help searching for flights to Paris? If so, I'll need to know which city you're traveli